In [1]:
import polars as pl
import mlflow
import mlflow.sklearn
from deltalake import DeltaTable, write_deltalake
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, roc_auc_score, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
import os
import pandas as pd
import matplotlib.pyplot as plt
import tempfile

os.environ["GIT_PYTHON_REFRESH"] = "quiet"

storage_options = {
    "AWS_ACCESS_KEY_ID": os.getenv("AWS_ACCESS_KEY_ID", "admin"),
    "AWS_SECRET_ACCESS_KEY": os.getenv("AWS_SECRET_ACCESS_KEY", "password"),
    "AWS_ENDPOINT_URL": os.getenv("S3_ENDPOINT", "http://minio:9000"),
    "AWS_REGION": "us-east-1",
    "AWS_ALLOW_HTTP": "true"
}

base_path = "s3://lakehouse"

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://mlflow:5000"))

exp_name = "Flight_Delays_Prediction"

if not mlflow.get_experiment_by_name(exp_name):
    mlflow.create_experiment(
        name=exp_name,
        artifact_location="s3://mlflow/" 
    )

mlflow.set_experiment(exp_name)
print(f"Experiment set to: {exp_name}")

Experiment set to: Flight_Delays_Prediction


Партицирование по году и месяцу является разумным подходом, так как аналитические запросы к истории полетов чаще всего относятся к временным рамкам.

Вывод `.explain()` для цепочки:

In [2]:
query = (
    pl.scan_delta(f"{base_path}/silver", storage_options=storage_options)
    .filter(pl.col("Year") == 2024)
    .filter(pl.col("Origin") == "JFK")
    .select(["FlightDate", "Origin", "ArrDelay"])
)

print(query.explain())

simple π 3/3 ["FlightDate", "Origin", ... 1 other column]
  Parquet SCAN [s3://lakehouse/silver/Year=2024/Month=1/part-00000-52d256a2-299e-4ea5-bcb3-796f129b9441-c000.zstd.parquet]
  PROJECT 4/13 COLUMNS
  SELECTION: [([(col("Origin")) == ("JFK")]) & ([(col("Year")) == (2024)])]


Видно, что поиск велся только в одной папке, соответствующей году запроса, также читались не все колонки в таблице, а только необходимые.

Далее показаны возможности по Time Travel и Schema Evolution:

In [3]:
dt_silver = DeltaTable(f"{base_path}/silver", storage_options=storage_options)
print(f"Текущая версия Silver таблицы: {dt_silver.version()}")

df_v0 = pl.scan_delta(
    f"{base_path}/silver", 
    version=0, 
    storage_options=storage_options
).collect()
print(f"Размер таблицы в версии 0: {df_v0.shape}")

df_gold = pl.scan_delta(f"{base_path}/gold_features", storage_options=storage_options).collect()
df_gold_evolved = df_gold.with_columns(
    pl.col("DayOfWeek").is_in([6, 7]).cast(pl.Int32).alias("Is_Weekend")
)

write_deltalake(
    f"{base_path}/gold_features", 
    df_gold_evolved, 
    mode="append", 
    schema_mode="merge", 
    storage_options=storage_options
)
print("'Is_Weekend' успешно добавлена в схему Delta Table")

Текущая версия Silver таблицы: 1
Размер таблицы в версии 0: (558388, 13)
'Is_Weekend' успешно добавлена в схему Delta Table


Далее представлен код по выгрузке витрины с признаками для обучения моделей регрессии и классификации, кодирования категориальных признаков, разбиения данных на тренировочную и тестовую выборки:

In [ ]:
gold_dt = DeltaTable(f"{base_path}/gold_features", storage_options=storage_options)
df = pl.scan_delta(f"{base_path}/gold_features", storage_options=storage_options).collect()

df_pandas = df.to_pandas()
df_encoded = pd.get_dummies(df_pandas, columns=["Season", "Origin", "Dest"], drop_first=True)

X = df_encoded.drop(columns=["ArrDelay"])
y_reg = df_encoded["ArrDelay"]
y_clf = (df_encoded["ArrDelay"] > 15).astype(int)

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2)

Далее представлен код функции, обучающий модели и фиксирующий информацию о процессе обучения:

In [ ]:
def train_and_log_models():
    with mlflow.start_run(run_name="Random_Forest_Regression"):
        mlflow.log_param("gold_table_version", gold_dt.version())
        
        print("Обучение модели регрессии...")
        reg_params = {"n_estimators": 50, "max_depth": 10}
        reg = RandomForestRegressor(**reg_params)
        reg.fit(X_train, y_reg_train)
        
        reg_preds = reg.predict(X_test)
        
        mse = mean_squared_error(y_reg_test, reg_preds)
        r2 = r2_score(y_reg_test, reg_preds)
        
        mlflow.log_params(reg_params)
        mlflow.log_metrics({"mse": mse, "r2": r2})
        mlflow.sklearn.log_model(reg, "model")

    with mlflow.start_run(run_name="Random_Forest_Classification"):
        mlflow.log_param("gold_table_version", gold_dt.version())
        
        print("Обучение модели классификации (RF)...")
        clf_params = {"n_estimators": 50, "max_depth": 10}
        clf = RandomForestClassifier(**clf_params)
        clf.fit(X_train, y_clf_train)
        
        clf_preds = clf.predict(X_test)
        clf_probs = clf.predict_proba(X_test)[:, 1]
        
        acc = accuracy_score(y_clf_test, clf_preds)
        auc = roc_auc_score(y_clf_test, clf_probs)
        
        mlflow.log_params(clf_params)
        mlflow.log_metrics({"accuracy": acc, "roc_auc": auc})

        report = classification_report(y_clf_test, clf_preds)
        print(report)
        with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
            f.write(report)
            mlflow.log_artifact(f.name, "classification_report.txt")

        fig, ax = plt.subplots(figsize=(8, 6))
        ConfusionMatrixDisplay.from_estimator(clf, X_test, y_clf_test, ax=ax, cmap="Blues")
        plt.title("RF Confusion Matrix")
        fig.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close(fig)

        fi_df = pd.DataFrame({"Feature": X.columns, "Importance": clf.feature_importances_}).sort_values("Importance", ascending=False)
        fig, ax = plt.subplots(figsize=(10, 6))
        fi_df.plot(kind="barh", x="Feature", y="Importance", ax=ax)
        plt.title("Feature Importance")
        fig.savefig("feature_importance.png")
        mlflow.log_artifact("feature_importance.png")
        plt.close(fig)
        
        mlflow.sklearn.log_model(clf, "model")

    with mlflow.start_run(run_name="Logistic_Classification"):
        mlflow.log_param("gold_table_version", gold_dt.version())
        
        print("Обучение модели классификации (LR)...")
        clf_lr_params = {"max_iter": 500}
        lr = LogisticRegression(**clf_lr_params)
        lr.fit(X_train, y_clf_train)
        
        lr_preds = lr.predict(X_test)
        lr_probs = lr.predict_proba(X_test)[:, 1]
        
        acc = accuracy_score(y_clf_test, lr_preds)
        auc = roc_auc_score(y_clf_test, lr_probs)
        
        mlflow.log_params(clf_lr_params)
        mlflow.log_metrics({"accuracy": acc, "roc_auc": auc})

        report = classification_report(y_clf_test, lr_preds)
        print(report)
        with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
            f.write(report)
            mlflow.log_artifact(f.name, "classification_report.txt")

        fig, ax = plt.subplots(figsize=(8, 6))
        ConfusionMatrixDisplay.from_estimator(lr, X_test, y_clf_test, ax=ax, cmap="Blues")
        plt.title("LR Confusion Matrix")
        fig.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close(fig)

        mlflow.sklearn.log_model(lr, "model")

In [ ]:
train_and_log_models()

Сравнить модели, посмотреть графики и другую информацию можно в MLFlow (0.0.0.0:5000)